# Chapter 11 Training Deep Neural Networks
## Section 4 Avoiding Overfitting Through Regularization
### Section 4.1 L1 and L2 Regularization

#### Experiment Setting

In [1]:
import time
from pathlib import Path

import numpy as np
import tensorflow.keras as keras

##### Dataset

In [2]:
# Load dataset
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()
class_names = np.array(
    [
        "T-shirt/top",
        "Trouser",
        "Pullover",
        "Dress",
        "Coat",
        "Sandal",
        "Shirt",
        "Sneaker",
        "Bag",
        "Ankle boot",
    ]
)

# Normalize data
X_train, X_valid = X_train_full[:-5000] / 255.0, X_train_full[-5000:] / 255.0
X_test = X_test / 255.0
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]

# Inspect data
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape}")
print(f"y_valid: {y_valid.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

X_train: (55000, 28, 28)
y_train: (55000,)
X_valid: (5000, 28, 28)
y_valid: (5000,)
X_test: (10000, 28, 28)
y_test: (10000,)


##### TensorBoard Setting

In [3]:
%load_ext tensorboard
%tensorboard --logdir=./my_logs --port=6006

Launching TensorBoard...

##### Hyperparameters

In [4]:
epochs = 50
batch = 32
optimizer = keras.optimizers.SGD()
loss = "sparse_categorical_crossentropy"
metric = "accuracy"

##### Putting all together

In [5]:
class Experiment:
    def __init__(
        self,
        model: keras.Model,
        optimizer: keras.optimizers.Optimizer = optimizer,
        loss: str = loss,
        metric: str = metric,
        epochs: int = epochs,
    ):
        self.model: keras.Model = model
        self.optimizer: keras.optimizers.Optimizer = optimizer
        self.loss: str = loss
        self.metric: str = metric
        self.epochs: int = epochs
        self.dir_log_run: str = ""

        self.setup_log_dir()
        self.compile()

    def setup_log_dir(self):
        run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
        self.dir_log_run = str(Path() / "my_logs" / run_id)

    def compile(self) -> None:
        self.model.compile(
            loss=self.loss, optimizer=self.optimizer, metrics=[self.metric]
        )

    def fit(
        self,
        X_train: np.ndarray = X_train,
        y_train: np.ndarray = y_train,
        X_valid: np.ndarray = X_valid,
        y_valid: np.ndarray = y_valid,
    ) -> keras.callbacks.History:
        tensorboard_cb = keras.callbacks.TensorBoard(self.dir_log_run)
        return self.model.fit(
            X_train,
            y_train,
            epochs=self.epochs,
            validation_data=(X_valid, y_valid),
            callbacks=[tensorboard_cb],
        )

    def evaluate(
        self, X_test: np.ndarray = X_test, y_test: np.ndarray = y_test
    ) -> None:
        print(
            f"{self.model.metrics_names[0]}: {self.model.evaluate(X_test, y_test)[0]}\n{self.model.metrics_names[1]}: {self.model.evaluate(X_test, y_test)[1]}"
        )

#### Experiment 1: no regularization

In [6]:
model_base = keras.Sequential(
    [
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(300, activation="selu", kernel_initializer="lecun_normal"),
        keras.layers.Dense(100, activation="selu", kernel_initializer="lecun_normal"),
        keras.layers.Dense(len(class_names), activation="softmax"),
    ]
)
model_base.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
flatten (Flatten)            (None, 784)               0         
_________________________________________________________________
dense (Dense)                (None, 300)               235500    
_________________________________________________________________
dense_1 (Dense)              (None, 100)               30100     
_________________________________________________________________
dense_2 (Dense)              (None, 10)                1010      
Total params: 266,610
Trainable params: 266,610
Non-trainable params: 0
_________________________________________________________________


In [7]:
exp = Experiment(model=model_base)
exp.fit()

Epoch 1/50


InternalError:  Blas GEMM launch failed : a.shape=(32, 784), b.shape=(784, 300), m=32, n=300, k=784
	 [[node sequential/dense/MatMul (defined at <ipython-input-5-ef6f7571735d>:33) ]] [Op:__inference_train_function_507]

Function call stack:
train_function


In [ ]:
exp.evaluate()

#### Experiment 2: L2 regression

In [ ]:
from functools import partial

RegularizedDense = partial(
    keras.layers.Dense,
    activation="selu",
    kernel_initializer="lecun_normal",
    kernel_regularizer=keras.regularizers.l2(0.01),
)
model_l2 = keras.Sequential(
    [
        keras.layers.Flatten(input_shape=(28, 28)),
        RegularizedDense(300),
        RegularizedDense(100),
        keras.layers.Dense(len(class_names), activation="softmax"),
    ]
)

In [ ]:
exp = Experiment(model=model_l2)
history = exp.fit()

In [ ]:
exp.evaluate()

#### Experiment 3: L1 regression

In [ ]:
from functools import partial

RegularizedDense = partial(
    keras.layers.Dense,
    activation="selu",
    kernel_initializer="lecun_normal",
    kernel_regularizer=keras.regularizers.l1(0.01),
)
model_l1 = keras.Sequential(
    [
        keras.layers.Flatten(input_shape=(28, 28)),
        RegularizedDense(300),
        RegularizedDense(100),
        keras.layers.Dense(len(class_names), activation="softmax"),
    ]
)

In [ ]:
exp = Experiment(model=model_l1)
history = exp.fit()

In [ ]:
exp.evaluate()

#### Experiment 4: Dropout

In [ ]:
dropout_rate = 0.2
model_dropout = keras.Sequential(
    [
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.AlphaDropout(
            dropout_rate
        ),  # Use alpha dropout for self-normalizing network based on the SELU activation function
        keras.layers.Dense(300, activation="selu", kernel_initializer="lecun_normal"),
        keras.layers.AlphaDropout(dropout_rate),
        keras.layers.Dense(100, activation="selu", kernel_initializer="lecun_normal"),
        keras.layers.AlphaDropout(dropout_rate),
        keras.layers.Dense(len(class_names), activation="softmax"),
    ]
)

In [ ]:
exp = Experiment(model=model_dropout)
history = exp.fit()

In [ ]:
exp.evaluate()

In [ ]:
# Metrics calculation by hand
from tensorflow.keras.metrics import (
    sparse_categorical_crossentropy,
    sparse_categorical_accuracy,
)

y_prob = model_dropout.predict(X_test)
loss_dropout = np.mean(sparse_categorical_crossentropy(y_test, y_prob))
accuracy_dropout = sum(sparse_categorical_accuracy(y_test, y_prob)) / len(y_test)

print(f"loss: {loss_dropout}\naccuracy: {accuracy_dropout}")

#### Experiment 5: Monte Carlo (MC) Dropout

In [ ]:
class AlphaMCDropout(keras.layers.AlphaDropout):
    def call(self, inputs):
        return super().call(inputs, training=True)


model_mcdropout = keras.Sequential(
    [
        layer
        if not isinstance(layer, keras.layers.AlphaDropout)
        else AlphaMCDropout(dropout_rate)
        for layer in model_dropout.layers
    ]
)
model_mcdropout.compile(loss=loss, optimizer=optimizer, metrics=[metric])

In [ ]:
y_probs = np.array([model_mcdropout.predict(X_test) for instance in range(100)])
y_prob = y_probs.mean(axis=0)
y_std = y_probs.std(axis=0)

loss_mcdropout = np.mean(sparse_categorical_crossentropy(y_test, y_prob))
accuracy_mcdropout = sum(sparse_categorical_accuracy(y_test, y_prob)) / len(y_test)

In [ ]:
print(f"loss: {loss_mcdropout}\naccuracy: {accuracy_mcdropout}")

#### Experiment 6: Max-Norm Regularization

In [ ]:
MaxNormDense = partial(
    keras.layers.Dense,
    activation="selu",
    kernel_initializer="lecun_normal",
    kernel_constraint=keras.constraints.max_norm(1.0),
)
model_maxnorm = keras.Sequential(
    [
        keras.layers.Flatten(input_shape=(28, 28)),
        MaxNormDense(300),
        MaxNormDense(100),
        keras.layers.Dense(len(class_names), activation="softmax"),
    ]
)

In [ ]:
exp = Experiment(model=model_maxnorm)
history = exp.fit()

In [ ]:
exp.evaluate()